# 🎓 FaceTrack AI — Google Colab Testing

Run the **full** Face Recognition Attendance System on Google Colab with **NVIDIA GPU acceleration** and the **complete web UI**.

---

## ⚠️ Before you start — Set runtime to T4 GPU

> `Runtime` → `Change runtime type` → `Hardware accelerator` → **T4 GPU** → Save

---

## 🚀 One-click start

Press **`Ctrl + F9`** (or `Runtime → Run all`) to run every cell automatically.  
The Cleanup cell at the bottom is safely guarded — it won't stop anything unless you set `_AUTO_STOP = True`.

---

## How it works

| Component | Local PC | Google Colab |
|---|---|---|
| **Camera** | USB webcam via `cv2.VideoCapture` | `/colab-camera` page — browser webcam, no notebook tab needed |
| **Inference** | OpenVINO CPU EP | `onnxruntime-gpu` → CUDA EP (NVIDIA T4) |
| **Display** | MJPEG stream | Canvas overlay on live video (smooth 30fps + 5fps GPU boxes) |
| **Web UI** | `http://localhost:5000` | Cloudflare Tunnel public URL (no account needed) |

In [ ]:
# ✅ Cell 1 — Verify GPU is connected
# ─────────────────────────────────────────────────────────────────────────────
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU detected:\n")
    print(result.stdout)
else:
    print("❌  NO GPU DETECTED!")
    print("─" * 60)
    print("Go to:  Runtime  ›  Change runtime type")
    print("Set 'Hardware accelerator' to  T4 GPU  then save.")
    print("Then:  Runtime  ›  Restart session  and re-run all cells.")
    print("─" * 60)
    raise SystemExit("Enable GPU before continuing.")

In [ ]:
%%bash
# 📦 Cell 2 — Install Colab-specific dependencies
set -e

echo "► Installing system libraries..."
apt-get install -qq libgl1 libglib2.0-0 2>/dev/null | tail -1

echo "► Installing cloudflared (zero-setup HTTPS tunnel — no account needed)..."
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
     -O /tmp/cloudflared.deb
dpkg -i /tmp/cloudflared.deb | tail -2

echo "► Installing Python packages..."
pip install -q \
    insightface \
    onnxruntime-gpu \
    onnx \
    "opencv-contrib-python-headless>=4.9.0" \
    "scikit-learn>=1.4.0" \
    "flask>=3.0.0" \
    "flask-socketio>=5.3.6" \
    "waitress>=3.0.0" \
    "python-dotenv>=1.0.1" \
    "flask-limiter>=3.5.0" \
    "Werkzeug>=3.0.1" \
    "reportlab>=4.1.0" \
    "openpyxl>=3.1.2" \
    "pandas>=2.2.0" \
    "Pillow>=10.2.0" \
    "numpy>=1.26.0"

echo ""
echo "✅ All dependencies installed."

In [ ]:
# 📂 Cell 3 — Clone repository and build the Vue frontend
# frontend/dist/ is gitignored and must be built here after cloning.
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess

REPO_URL = "https://github.com/Abhirup-1234/Face-Recognition-Attendance-System.git"
REPO_DIR = "/content/FaceTrack"

# ── Clone or update ───────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✅ Repository cloned.")
else:
    print("Repository exists — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("✅ Repository up to date.")

# ── Create .env ───────────────────────────────────────────────────────────────
with open(f"{REPO_DIR}/.env", "w") as f:
    f.write(
        "SECRET_KEY=colab-test-secret-key-9f4a2b8c3d1e\n"
        "ADMIN_USERNAME=admin\n"
        "ADMIN_PASSWORD=admin123\n"
    )
print("Credentials: admin / admin123")

# ── Build the Vue frontend ────────────────────────────────────────────────────
# Flask's _serve_spa() needs frontend/dist/index.html to exist.
FRONTEND_DIR = f"{REPO_DIR}/frontend"
DIST_INDEX   = f"{FRONTEND_DIR}/dist/index.html"

if not os.path.exists(DIST_INDEX):
    print("\n🔨 Building Vue frontend (npm install + npm run build)...")
    print("   Takes about 1-2 minutes on first run.\n")
    subprocess.run(["npm", "install"], cwd=FRONTEND_DIR, check=True)
    subprocess.run(["npm", "run", "build"], cwd=FRONTEND_DIR, check=True)
    print("\n✅ Frontend built.")
else:
    print("\n✅ Frontend already built — skipping.")

print(f"\n✅ Ready. Run the next cell.")

In [ ]:
# ⚙️ Cell 4 — Configure for Colab and start Flask
# ─────────────────────────────────────────────────────────────────────────────
import sys, os, threading, time, webbrowser

REPO_DIR = "/content/FaceTrack"

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.chdir(REPO_DIR)

webbrowser.open = lambda url, **kw: None  # no GUI on Colab VM

# ── Patch config BEFORE importing app ────────────────────────────────────────
import config

# Push camera — PushCameraProcessor activates when source starts with "push://"
config.CAMERAS = {"CAM-COLAB": "push://CAM-COLAB"}

# Allow all origins (Cloudflare tunnel URL + Colab notebook iframe)
config.CORS_ORIGINS = "*"

print(f"BASE_DIR : {config.BASE_DIR}")
print(f"CAMERAS  : {config.CAMERAS}")

# ── Import app (SocketIO reads CORS_ORIGINS at import time) ──────────────────
print("\n🔄 Loading app... (InsightFace buffalo_l models ~500 MB on first run)")
import app as app_module
from face_engine import FaceEngine
import onnxruntime as ort

# ── Verify CUDA and patch FaceEngine ─────────────────────────────────────────
eps = ort.get_available_providers()
print(f"\nONNX Runtime {ort.__version__} — available providers: {eps}")

if "CUDAExecutionProvider" in eps:
    print("✅ CUDAExecutionProvider available — NVIDIA GPU will be used!")
else:
    print("⚠️  CUDA not found — will use CPU. Enable T4 GPU in Runtime settings.")


def _colab_select_providers():
    available = ort.get_available_providers()
    if "CUDAExecutionProvider" in available:
        print("[FaceEngine] ✅ Using CUDAExecutionProvider")
        return ["CUDAExecutionProvider", "CPUExecutionProvider"]
    print("[FaceEngine] ⚠️  Using CPUExecutionProvider (CUDA unavailable)")
    return ["CPUExecutionProvider"]


FaceEngine._select_providers = staticmethod(_colab_select_providers)
FaceEngine._instance = None  # reset singleton — re-init with CUDA on next call

# ── Create Flask app + CORS ───────────────────────────────────────────────────
flask_app = app_module.create_app()

from flask import request as _flask_req


@flask_app.before_request
def _preflight():
    if _flask_req.method == "OPTIONS":
        r = flask_app.make_default_options_response()
        r.headers["Access-Control-Allow-Origin"]  = "*"
        r.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS, DELETE, PUT"
        r.headers["Access-Control-Allow-Headers"] = "Content-Type"
        return r


@flask_app.after_request
def _cors(response):
    response.headers["Access-Control-Allow-Origin"]  = "*"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS, DELETE, PUT"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type"
    return response


# ── Start Flask in background thread ─────────────────────────────────────────
_ready = threading.Event()


def _run():
    app_module.startup(flask_app)
    _ready.set()
    app_module.socketio.run(
        flask_app, host="0.0.0.0", port=5000,
        use_reloader=False, log_output=False,
        allow_unsafe_werkzeug=True,
    )


threading.Thread(target=_run, daemon=True).start()

print("\n⏳ Waiting for Flask (model download can take a few minutes on first run)...")
if _ready.wait(timeout=300):
    print("\n✅ Flask running on http://localhost:5000")
    print("▶ Run the next cell to start the Cloudflare Tunnel.")
else:
    print("❌ Flask didn't start in time. Check error output above.")

In [ ]:
# 🌐 Cell 5 — Start Cloudflare Tunnel (no account, no token)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, threading, re, time

TUNNEL_URL   = None
_tunnel_proc = None


def _run_tunnel():
    global TUNNEL_URL, _tunnel_proc
    _tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in _tunnel_proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
            break
    for _ in _tunnel_proc.stdout:
        pass   # keep tunnel process alive


threading.Thread(target=_run_tunnel, daemon=True).start()

print("⏳ Starting Cloudflare Tunnel (5-15 seconds)...")
for _ in range(45):
    if TUNNEL_URL:
        break
    time.sleep(1)

if TUNNEL_URL:
    print(f"\n{'=' * 64}")
    print(f"  ✅  FaceTrack AI is LIVE on Colab GPU!")
    print(f"{'=' * 64}")
    print(f"\n  🌐  Web UI (enroll, attendance, reports):")
    print(f"      {TUNNEL_URL}")
    print(f"\n  📷  Camera page (open this to stream your webcam):")
    print(f"      {TUNNEL_URL}/colab-camera")
    print(f"\n  🔑  Login: admin / admin123")
    print(f"\n  ▶   Run the next cell to print a clickable summary.")
    print(f"{'=' * 64}")
else:
    print("❌ Timed out. Make sure Flask started (Cell 4 completed successfully).")
    print("   Try running this cell again.")

In [ ]:
# 🔗 Cell 6 — Clickable links summary
# ─────────────────────────────────────────────────────────────────────────────
# The /colab-camera page is the camera page — it runs entirely in the
# Cloudflare URL (no notebook tab needed). It:
#   • Captures your browser webcam locally (smooth 30fps video)
#   • Sends frames to the Colab GPU for face detection (5fps)
#   • Draws bounding boxes as a canvas overlay — no MJPEG stream lag
#   • Attendance is still marked correctly via the background processor
# ─────────────────────────────────────────────────────────────────────────────
from IPython.display import HTML, display
import config as _cfg

if not TUNNEL_URL:
    print("❌  TUNNEL_URL not set. Run Cell 5 first.")
else:
    CAM_PAGE = TUNNEL_URL + "/colab-camera"

    display(HTML(
        '<div style="font-family:system-ui;padding:16px;background:#0d1117;'
        'border-radius:10px;border:1px solid #30363d;max-width:520px">'
        '<h3 style="color:#3fb950;margin:0 0 12px">&#128640; FaceTrack AI is ready!</h3>'

        '<p style="margin:6px 0;font-size:13px;color:#c9d1d9">'
        '<b>&#128247; Camera + Face Detection:</b></p>'
        '<a href="' + CAM_PAGE + '" target="_blank" '
        'style="font-size:14px;color:#58a6ff;word-break:break-all">'
        + CAM_PAGE + '</a>'
        '<p style="margin:2px 0 10px;font-size:11px;color:#484f58">'
        'Open this page → allow camera → see GPU detection with no lag.</p>'

        '<p style="margin:6px 0;font-size:13px;color:#c9d1d9">'
        '<b>&#127968; Full Web UI (enroll, reports, settings):</b></p>'
        '<a href="' + TUNNEL_URL + '" target="_blank" '
        'style="font-size:14px;color:#58a6ff;word-break:break-all">'
        + TUNNEL_URL + '</a>'
        '<p style="margin:2px 0 10px;font-size:11px;color:#484f58">'
        'Login: <b>admin</b> / <b>admin123</b> — enables recognition from the Cameras tab.</p>'

        '<hr style="border-color:#30363d;margin:10px 0">'
        '<p style="font-size:11px;color:#484f58;margin:0">'
        '&#9888;&#65039;  Attendance marking requires recognition to be enabled <br>'
        'from the main web UI: Cameras tab &#8594; Enable Recognition.</p>'
        '</div>'
    ))

    print(f"\nCamera page : {CAM_PAGE}")
    print(f"Web UI      : {TUNNEL_URL}")
    print(f"Login       : admin / admin123")

---
## 🛑 Cleanup — run only when you want to stop

The cell below **does nothing** by default, so `Runtime → Run all` is completely safe.  
Change `_AUTO_STOP = True` and run just this cell when you're done.

In [ ]:
# 🛑 Cleanup — set _AUTO_STOP = True and run this cell to stop the tunnel
# Safe to include in Runtime → Run all (does nothing while _AUTO_STOP is False)
_AUTO_STOP = False   # ← change to True when you want to stop

if _AUTO_STOP:
    print("Stopping Cloudflare Tunnel...")
    try:
        _tunnel_proc.terminate()
        print("✅ Tunnel stopped.")
    except Exception as e:
        print(f"⚠️  {e}")
    print("\nFlask and GPU threads stop when you disconnect the runtime.")
    print("Go to:  Runtime  ›  Disconnect and delete runtime")
else:
    print("ℹ️  Cleanup skipped (_AUTO_STOP is False).")
    print("   Change to True and re-run this cell when done.")